# Chilbolton Climatology — Student 1: Rainfall

This is **Student 1's notebook** as part of a group project on the climatology of Chilbolton Observatory.
The four students are each analysing a different meteorological variable:
- **Student 1 (you)**: Rainfall
- **Student 2**: Air temperature and relative humidity
- **Student 3**: Surface pressure
- **Student 4**: Wind speed and direction

## Learning Objectives
By the end of this notebook you should be able to:
- Load rainfall data from NetCDF files and apply QC flags
- Aggregate high-frequency tip/drop counts into daily, monthly, and annual accumulations
- Compute and interpret climatological statistics for precipitation
- Plot rainfall time series at multiple time scales
- Construct and interpret a rainfall exceedance curve

## Instrument Information
Rainfall at Chilbolton is measured by tipping-bucket or disdrometer rain gauges.
- The data variable is `thickness_of_rainfall_amount`, stored in **mm** per sampling interval
- Some older files store tip counts (`number_of_tips`); each tip = 0.2 mm
- A single QC flag variable (`qc_flag`) is provided for each file
- QC flag values: **1 = good data**, anything else = suspect or bad — always filter before analysis
- The sampling interval is approximately **10 seconds** throughout

## Setup — Run This First

This cell loads the necessary libraries and defines helper functions you will use throughout the notebook. You do not need to modify it.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import re

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import netCDF4 as nc4
import numpy as np
import pandas as pd

try:
    import cftime
except ImportError:
    cftime = None

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)


@dataclass
class GaugeData:
    """Container for one gauge's raw data arrays."""
    label:       str
    source:      str
    physical_id: str
    unix:        np.ndarray   # Unix timestamps (s since 1970-01-01)
    mm:          np.ndarray   # Rainfall per sample (mm)
    qc:          np.ndarray   # QC flag (int8)
    file_count:  int


# ── Low-level helpers ────────────────────────────────────────────────────────

def median_dt(unix):
    """Return the median time step (seconds) of a Unix-time array."""
    if len(unix) < 2:
        return 10.0
    diffs = np.diff(unix[: min(len(unix), 2000)])
    diffs = diffs[(diffs > 0) & (diffs < 3600)]
    return float(np.median(diffs)) if len(diffs) else 10.0

def bad_mask(qc):
    """Return a boolean mask that is True where data are flagged as bad."""
    return (qc != 0) & (qc != 1)

def apply_qc(mm, qc, qc_good_only):
    """Return a copy of mm with bad-flagged values replaced by NaN (if qc_good_only)."""
    out = mm.astype(float).copy()
    if qc_good_only:
        out[bad_mask(qc)] = np.nan
    return out

def infer_label(stem):
    m = re.search(r'(?:ncas|stfc)-rain-gauge-\d+', stem.lower())
    if m:
        return m.group(0)
    if 'cfarr-multiple-raingauges' in stem.lower():
        return 'cfarr-multiple-raingauges'
    return stem.split('_cao_')[0] if '_cao_' in stem else stem.split('_')[0]

def infer_source(path, label):
    ll, pp = label.lower(), str(path).lower()
    if ll.startswith('stfc-') or '/stfc/' in pp: return 'stfc'
    if ll.startswith('ncas-') or '/ncas/' in pp: return 'ncas'
    if 'cfarr' in ll or 'cfarr' in pp:           return 'cfarr'
    return 'other'

def infer_physical_id(label):
    m = re.search(r'rain-gauge-(\d+)', label.lower())
    if m: return f'rg{int(m.group(1))}'
    if 'cfarr-multiple-raingauges' in label.lower(): return 'rg1'
    return label.lower()

def read_nc_single(path):
    p = Path(path)
    with nc4.Dataset(str(p)) as nc:
        unix = nc.variables['time'][:].data.copy().astype(np.float64)
        time_units = ''
        if 'units' in nc.variables['time'].ncattrs():
            time_units = nc.variables['time'].getncattr('units')
        if time_units and cftime:
            times = cftime.num2date(unix, time_units)
            unix = np.array([pd.Timestamp(t.isoformat()).timestamp() for t in times], dtype=np.float64)
        if 'thickness_of_rainfall_amount' in nc.variables:
            mm = nc.variables['thickness_of_rainfall_amount'][:].data.copy().astype(np.float64)
        elif any(v in nc.variables for v in ('number_of_tips', 'number_of_drops', 'drop_count_a')):
            count_var = next(v for v in ('number_of_tips', 'number_of_drops', 'drop_count_a') if v in nc.variables)
            counts = nc.variables[count_var][:].data.copy().astype(np.float64)
            quanta = 0.2 if count_var == 'number_of_tips' else 0.004
            if 'measurement_quanta' in nc.ncattrs():
                try: quanta = float(str(nc.getncattr('measurement_quanta')).split()[0])
                except (ValueError, IndexError): pass
            mm = counts * quanta
        else:
            raise ValueError(f'No rainfall variable found in {p.name}')
        qc = nc.variables['qc_flag'][:].data.copy().astype(np.int8) if 'qc_flag' in nc.variables else np.zeros(len(unix), dtype=np.int8)
    label = infer_label(p.stem)
    return {'unix': unix, 'mm': mm, 'qc': qc,
            'label': label,
            'source': infer_source(p, label),
            'physical_id': infer_physical_id(label)}


def load_gauges(root, source_filter='all', merge_by_physical=False):
    """
    Load all rain gauge NetCDF files from root.

    Parameters
    ----------
    root              : path to the precipitation data directory
    source_filter     : 'all', 'ncas', 'stfc', or 'cfarr'
    merge_by_physical : if True, merge files from the same physical gauge

    Returns
    -------
    dict mapping gauge key → GaugeData
    """
    p = Path(root).expanduser().resolve()
    files = sorted(f for f in p.rglob('*.nc'))
    if source_filter != 'all':
        files = [f for f in files if f.stem.lower().startswith(f'{source_filter}-')]
    if not files:
        raise ValueError(f'No .nc files found for source={source_filter} under {root}')
    by_key = {}
    for f in files:
        d = read_nc_single(str(f))
        key = d['physical_id'] if merge_by_physical else d['label']
        by_key.setdefault(key, []).append(d)
    out = {}
    for key, chunks in sorted(by_key.items()):
        unix = np.concatenate([c['unix'] for c in chunks])
        mm   = np.concatenate([c['mm']   for c in chunks])
        qc   = np.concatenate([c['qc']   for c in chunks])
        order = np.argsort(unix, kind='stable')
        sources = sorted(set(c['source'] for c in chunks))
        out[key] = GaugeData(
            label=key if merge_by_physical else chunks[0]['label'],
            source=sources[0] if len(sources) == 1 else 'mixed',
            physical_id=chunks[0]['physical_id'],
            unix=unix[order], mm=mm[order], qc=qc[order],
            file_count=len(chunks))
    return out


print('Libraries loaded and helper functions defined.')

---
## Task 1: Load and Explore the Data

**What to do:**
1. Set `ROOT_PATH` to the precipitation data directory (already filled in below)
2. Run the cell to load the data — we will use gauge **rg1** throughout this notebook
3. Look at the printed output and answer the questions below

**Questions to answer in your report:**
- What is the date range of the record for rg1?
- What is the approximate time step (sampling interval) in seconds?
- What percentage of samples are flagged as bad quality?


In [ ]:
# ===== EDIT THESE IF NEEDED =====
ROOT_PATH        = '/data/wexp/cwalden/precipitation'
MERGE_BY_PHYSICAL = True   # True = merge files from the same physical gauge
QC_GOOD_ONLY     = True    # True = discard bad-flagged data
PLOT_THEME       = 'light'  # 'dark' or 'light'
# ================================

plt.style.use('dark_background' if PLOT_THEME == 'dark' else 'default')

gauges = load_gauges(ROOT_PATH, merge_by_physical=MERGE_BY_PHYSICAL)

# We will focus on gauge rg1 throughout this notebook
g = gauges['rg1']

dt_s    = median_dt(g.unix)
t_start = pd.Timestamp(g.unix[0],  unit='s').strftime('%Y-%m-%d')
t_end   = pd.Timestamp(g.unix[-1], unit='s').strftime('%Y-%m-%d')
n_bad   = int(bad_mask(g.qc).sum())
print(f'  {g.label:<12}  source={g.source:<6}  files={g.file_count:5d}'
      f'  samples={len(g.unix):>10,}  dt={dt_s:.0f}s'
      f'  {t_start} → {t_end}'
      f'  bad={100*n_bad/len(g.qc):.1f}%')


---
## Task 2: Summary Statistics

**What to do:**
Complete the function `compute_summary_stats()` below. It should return a dictionary containing:

| Key | Description |
|---|---|
| `total_mm` | Total accumulated rainfall (mm) over the whole record |
| `mean_rate_mm_hr` | Mean rainfall rate (mm hr⁻¹) across all samples |
| `max_rate_mm_hr` | Maximum rainfall rate (mm hr⁻¹) in any single sample |
| `wet_fraction_pct` | Percentage of samples with rainfall > 0.01 mm hr⁻¹ |

**Hints:**
- Use `median_dt(g.unix)` to get the time step `dt_s` in seconds
- Rainfall rate in mm hr⁻¹ = `mm_per_sample * (3600 / dt_s)`
- Use `apply_qc(g.mm, g.qc, qc_good_only)` to get QC-filtered values
- Always clip negative values to zero with `np.maximum(array, 0.0)` — negative rainfall is non-physical
- Use `np.nanmean`, `np.nanmax`, `np.nansum` to ignore NaN values


In [ ]:
def compute_summary_stats(g: GaugeData, qc_good_only: bool = True) -> dict:
    """
    Return a dict of summary statistics for one gauge.
    """
    dt_s = median_dt(g.unix)
    work = apply_qc(g.mm, g.qc, qc_good_only)

    # TODO: compute rates in mm/hr
    # rates = ...

    # TODO: compute the statistics listed in the task description
    # total_mm       = ...
    # mean_rate_mm_hr = ...
    # max_rate_mm_hr  = ...
    # wet_fraction_pct = ...

    return {
        'label':            g.label,
        'source':           g.source,
        'date_start':       pd.Timestamp(g.unix[0],  unit='s').strftime('%Y-%m-%d'),
        'date_end':         pd.Timestamp(g.unix[-1], unit='s').strftime('%Y-%m-%d'),
        'total_mm':         None,   # replace with your calculation
        'mean_rate_mm_hr':  None,
        'max_rate_mm_hr':   None,
        'wet_fraction_pct': None,
    }


print(compute_summary_stats(g, qc_good_only=QC_GOOD_ONLY))


---
## Task 3: Temporal Aggregations

The raw data has one value every ~10 seconds. For analysis and plotting we need to aggregate it into daily, monthly, and annual totals.

**What to do:**
Complete the three functions below. Each should return a DataFrame with a time-period column and a `total_mm` column.

**What are Unix times?**
Computers often store times as a single large number: the number of seconds elapsed since midnight on 1 January 1970 (known as the *Unix epoch*). For example, `1 000 000 000` seconds after the epoch is 9 September 2001 at 01:46 UTC. The `g.unix` array contains one such number for every 10-second sample in the dataset. To work with them as human-readable dates, we convert them with `pd.to_datetime(unix, unit='s', utc=True)`.

**Hints:**
- Convert Unix times to pandas Timestamps: `pd.to_datetime(unix, unit='s', utc=True)`
- Use `.dt.normalize()` to strip the time and keep only the date (for daily grouping)
- Use `.to_period('M')` for monthly grouping, `.dt.year` for annual grouping
- Use `groupby(...).apply(lambda x: float(np.nansum(x)))` to sum each group, ignoring NaN
- Always clip negative values to zero before summing

**What is a `lambda`?**
A `lambda` is a small, one-line anonymous function — useful when you need a simple function in a single place and don't want to define it separately with `def`. For example:

```python
# These two are equivalent:
lambda x: float(np.nansum(x))

def my_sum(x):
    return float(np.nansum(x))
```

Here, `groupby('day').apply(...)` calls whatever function you pass in once for each group (i.e. once per day), with `x` being that day's slice of data. The `lambda` version is just a compact way of writing that.


In [ ]:
def daily_accumulation(unix, mm, qc, qc_good_only=True):
    """
    Aggregate per-sample rainfall into daily totals.
    Returns DataFrame with columns: day, total_mm
    """
    work = apply_qc(mm, qc, qc_good_only)
    ts = pd.to_datetime(unix, unit='s', utc=True)

    # TODO: group by day and sum
    # Hint: df = pd.DataFrame({'day': ts.normalize(), 'mm': np.maximum(work, 0.0)})
    #       then groupby 'day'
    pass


def monthly_accumulation(unix, mm, qc, qc_good_only=True):
    """
    Aggregate per-sample rainfall into monthly totals.
    Returns DataFrame with columns: period (YYYY-MM string), total_mm
    """
    work = apply_qc(mm, qc, qc_good_only)
    # TODO: use ts.to_period('M') for grouping
    pass


def annual_accumulation(unix, mm, qc, qc_good_only=True):
    """
    Aggregate per-sample rainfall into annual totals.
    Returns DataFrame with columns: year, total_mm
    """
    work = apply_qc(mm, qc, qc_good_only)
    # TODO: group by year
    pass


# Test
g = list(gauges.values())[0]
daily   = daily_accumulation(g.unix, g.mm, g.qc, qc_good_only=QC_GOOD_ONLY)
monthly = monthly_accumulation(g.unix, g.mm, g.qc, qc_good_only=QC_GOOD_ONLY)
annual  = annual_accumulation(g.unix, g.mm, g.qc, qc_good_only=QC_GOOD_ONLY)

print('Daily (last 5 days):');  print(daily.tail())
print('\nMonthly (last 6 months):'); print(monthly.tail(6))
print('\nAnnual:'); print(annual)

---
## Task 4: Visualisation

**What to do:**
Create a 3-panel figure showing daily, monthly, and annual rainfall for one gauge.

- **Top panel**: bar chart of daily accumulation for the most recent 100 days
- **Middle panel**: bar chart of monthly accumulation for the full record
- **Bottom panel**: bar chart of annual accumulation for the full record

**Hints:**
- Use `ax.bar(x, y)` for bar charts
- For the daily panel, use `daily.tail(100)` to select the last 100 days
- Add axis labels (`ax.set_xlabel`, `ax.set_ylabel`) and a title (`ax.set_title`)
- Use `ax.grid(True, axis='y', alpha=0.3)` to add horizontal gridlines
- Rotate x-axis tick labels if they overlap: `ax.tick_params(axis='x', rotation=45)`

In [ ]:
# g is already set to rg1 from Task 1

# Re-compute aggregations for the selected gauge
daily   = daily_accumulation(g.unix, g.mm, g.qc, qc_good_only=QC_GOOD_ONLY)
monthly = monthly_accumulation(g.unix, g.mm, g.qc, qc_good_only=QC_GOOD_ONLY)
annual  = annual_accumulation(g.unix, g.mm, g.qc, qc_good_only=QC_GOOD_ONLY)

# TODO: create 3-panel figure
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Panel 1: daily — last 100 days
# axes[0].bar(...)

# Panel 2: monthly
# axes[1].bar(...)

# Panel 3: annual
# axes[2].bar(...)

fig.tight_layout()
plt.show()

---
## Task 5: Monthly Climatology

Total rainfall varies month-to-month because of natural year-to-year variability, but averaged over many years a seasonal pattern emerges — the *climatology*. 

**What to do:**
1. Add a `month` column to your `monthly` DataFrame using `pd.to_datetime(monthly['period']).dt.month`
2. Group by month and compute the **mean** and **standard deviation** of monthly totals
3. Plot the result as a bar chart with error bars showing ± 1 standard deviation

**Hints:**
```python
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly['month'] = pd.to_datetime(monthly['period']).dt.month
clim = monthly.groupby('month')['total_mm'].agg(['mean', 'std'])
```
- Use `ax.bar(x, clim['mean'], yerr=clim['std'], capsize=4)` to add error bars
- Set x-tick labels to `MONTH_NAMES`

**Questions:**
- Which months tend to be wettest? Which driest?
- Is the year-to-year variability (std) large or small compared with the seasonal signal (range of monthly means)?


In [ ]:
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# TODO: add a 'month' column and compute climatology
# monthly['month'] = pd.to_datetime(monthly['period']).dt.month
# clim = monthly.groupby('month')['total_mm'].agg(['mean', 'std'])
# clim.index = MONTH_NAMES

# TODO: plot as bar chart with error bars
# fig, ax = plt.subplots(figsize=(10, 5))
# x = np.arange(12)
# ax.bar(x, clim['mean'], yerr=clim['std'], capsize=4, alpha=0.8, label='Mean ± 1 std')
# ax.set_xticks(x); ax.set_xticklabels(MONTH_NAMES)
# ax.set_ylabel('Monthly rainfall (mm)')
# ax.set_title('Climatological monthly rainfall — rg1')
# ax.legend(); ax.grid(True, axis='y', alpha=0.3)
# fig.tight_layout(); plt.show()


---
## Task 6: Exceedance Curve

An **exceedance curve** shows the probability that the rainfall rate exceeds a given threshold. It is plotted on a log-log scale so that rare intense events are visible alongside common light rain.

**What to do:**
Complete the function `exceedance_curve()` below, which:
1. Aggregates the 10-second data into fixed-length windows of `integration_minutes` minutes
2. Converts each window total into a rainfall rate (mm hr⁻¹)
3. Sorts rates in descending order and assigns each a plotting position
4. Returns a DataFrame with columns `rate_mm_hr` and `exceedance`

Then plot the exceedance curve for rg1 on log-log axes.

**Hints:**
- Number of samples per window: `n_per = round(integration_minutes * 60 / dt_s)`
- Reshape into blocks: `block = work[:n_total].reshape(n_periods, n_per)` (truncate to a multiple of `n_per` first)
- Rate per block: `np.nansum(block, axis=1) / (n_per * dt_s / 3600.0)`
- Plotting position for rank $i$ (0-indexed): $P_i = (i + 0.5) / N$
- Use `ax.set_xscale('log')` and `ax.set_yscale('log')`
- Exclude windows with fewer than 80% valid samples (`min_coverage=0.8`)

In [ ]:
def exceedance_curve(unix, mm, qc, integration_minutes=60, qc_good_only=True, min_coverage=0.8):
    """
    Compute the exceedance curve for a rain gauge record.

    Returns a DataFrame with columns:
        rate_mm_hr  : rainfall rate (mm hr⁻¹), sorted descending
        exceedance  : probability that the rate exceeds this value
    """
    dt_s = median_dt(unix)
    work = apply_qc(mm, qc, qc_good_only)

    # TODO: compute n_per (samples per integration window)
    # n_per = ...

    # TODO: reshape work into (n_periods, n_per) blocks
    # n_periods = len(work) // n_per
    # block = ...

    # TODO: filter out windows with too many NaN values
    # valid_count = np.sum(~np.isnan(block), axis=1)
    # ok = valid_count >= min_coverage * n_per

    # TODO: compute rate for each window and build the exceedance array

    return pd.DataFrame({'rate_mm_hr': [], 'exceedance': []})


# Plot exceedance curve for rg1
fig, ax = plt.subplots(figsize=(10, 6))

exc = exceedance_curve(g.unix, g.mm, g.qc, integration_minutes=60, qc_good_only=QC_GOOD_ONLY)
if not exc.empty:
    ax.plot(exc['rate_mm_hr'], exc['exceedance'], label=g.label, linewidth=1.5)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Rainfall rate (mm hr⁻¹)')
ax.set_ylabel('Probability of exceedance')
ax.set_title('Exceedance curve — 60-minute integration')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
plt.show()

---
## Task 7 (Optional): Wet-Day Frequency and Seasonal Breakdown

A *wet day* is conventionally defined as a day with at least 1 mm of rainfall.

**What to do:**
1. From your `daily` DataFrame, define wet days (daily total ≥ 1 mm) and compute:
   - The **overall probability** of rain on a randomly chosen day at Chilbolton (express as a fraction and as a percentage)
   - The **monthly wet-day probability** — how does this vary through the year?
2. Plot the monthly wet-day probability as a bar chart
3. Compute the mean and 95th-percentile daily rainfall on wet days only

**Questions:**
- What is the probability of rain on a randomly chosen day at Chilbolton? Does this surprise you?
- In which months is it most likely to rain? Least likely?
- What is the 95th percentile of *wet-day* daily rainfall? What does this mean in practical terms?
- Does the seasonal pattern in *frequency* match the seasonal pattern in *total accumulation* from Task 5? If not, what does the difference tell you?


In [ ]:
# TODO: define a wet day (>= 1 mm) and compute wet-day probability
# Hint:
# daily['month'] = daily['day'].dt.month
# daily['wet'] = daily['total_mm'] >= 1.0

# Overall probability of rain on any given day:
# p_rain = daily['wet'].mean()
# print(f'P(rain on a given day) = {p_rain:.3f}  ({p_rain*100:.1f}%)')

# Monthly wet-day probability:
# wet_prob = daily.groupby('month')['wet'].mean()

# 95th percentile on wet days only:
# wet_days = daily.loc[daily['wet'], 'total_mm']
# p95_wet  = np.percentile(wet_days, 95)


---
## Task 8: Monthly Rainfall Intensity

Total accumulation tells you *how much* rain falls each month, but intensity tells you *how hard* it rains. A month with many light drizzle events might have the same total as one with a few intense convective showers — yet the two are physically very different.

You will investigate this using **two integration times**: 1 minute and 60 minutes. Because extreme convective bursts are brief, they appear much more clearly at shorter integration times.

**What to do:**
1. Write a helper function `blocks_by_month(g, integration_minutes)` that aggregates the 10-second data into fixed-length windows, converts each window to a rate (mm hr⁻¹), and returns a DataFrame with columns `year`, `month` and `rate`
2. Call your function for **both 1-minute and 60-minute** integration times
3. For each integration time, filter to **wet periods** (rate > 0.1 mm hr⁻¹) and produce a 2-panel figure:
   - Left panel: box plot of wet-period rates grouped by calendar month (log y-axis)
   - Right panel: line plot of the **50th, 90th, 99th and 99.9th percentile** rates by month
4. **Per-year heatmap:** build a 2-D matrix (rows = years, columns = months) of the **99th-percentile wet-period rate** and display it as a heatmap (use `ax.imshow` with `cmap='YlOrRd'`). Is the summer intensity peak consistent year-to-year, or does it vary strongly?

**Hints:**
```python
def blocks_by_month(g, integration_minutes, min_coverage=0.8):
    dt_s      = median_dt(g.unix)
    n_per     = max(1, round(integration_minutes * 60 / dt_s))
    period_hr = n_per * dt_s / 3600.0
    work      = apply_qc(g.mm, g.qc, qc_good_only=True)
    n_periods = len(work) // n_per
    block     = work[: n_periods * n_per].reshape(n_periods, n_per)
    ok        = np.sum(np.isfinite(block), axis=1) >= min_coverage * n_per
    rates     = np.maximum(np.nansum(block, axis=1)[ok], 0.0) / period_hr
    unix_mid  = g.unix[: n_periods * n_per].reshape(n_periods, n_per).mean(axis=1)[ok]
    ts        = pd.to_datetime(unix_mid, unit='s', utc=True)
    return pd.DataFrame({'year': ts.year, 'month': ts.month, 'rate': rates})
```

**Questions:**
- At the same exceedance level, does the 1-minute or 60-minute integration give a higher rate? Why?
- Which months have the highest 99th-percentile rate at 1-minute integration? Do those same months top the 60-minute percentile too?
- Is the summer convective signal more or less visible at 1-minute compared to 60-minute? What does this tell you about the time-scale of intense events?
- Which months have the highest *total* accumulation (from Task 5)? Is this the same as the months with the highest *intensity* here?
- Looking at the heatmap: is the summer peak in the 99th percentile present in every year, or only in some? What does this tell you about sampling uncertainty?


In [ ]:
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

def blocks_by_month(g, integration_minutes, min_coverage=0.8):
    """Aggregate g into fixed windows; return DataFrame with 'year', 'month' and 'rate' columns."""
    # TODO: implement using the hints above
    ...


# ── Part A: climatological box plot + percentile lines ────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for row, integ_min in enumerate([1, 60]):
    integ_label = f'{integ_min}-min integration'

    # TODO: call blocks_by_month and filter to wet periods
    df_all = ...  # blocks_by_month(g, integration_minutes=integ_min)
    df_wet = ...  # filter: rate > 0.1

    # --- Panel left: box plot (log y-scale) ---
    ax = axes[row, 0]
    grouped = [df_wet.loc[df_wet['month'] == m, 'rate'].values for m in range(1, 13)]
    # TODO: ax.boxplot(grouped, labels=MONTH_NAMES, showfliers=True, ...)
    # TODO: ax.set_yscale('log')
    ax.set_ylabel('Rainfall rate (mm hr⁻¹)')
    ax.set_title(f'Box plot — {integ_label}')
    ax.grid(True, axis='y', which='both', alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

    # --- Panel right: percentile lines (50th, 90th, 99th, 99.9th) ---
    ax = axes[row, 1]
    x = np.arange(12)
    # TODO: for each percentile, compute per-month value and call ax.plot(...)
    ax.set_xticks(x)
    ax.set_xticklabels(MONTH_NAMES)
    ax.set_ylabel('Rainfall rate (mm hr⁻¹)')
    ax.set_title(f'Percentile intensity — {integ_label}')
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle(f'Monthly rainfall intensity — {g.label}', fontsize=13)
fig.tight_layout()
plt.show()

# ── Part B: heatmap of 99th-percentile by year and month ─────────────────────
YEAR_PCT   = 99   # percentile to show in the heatmap
INTEG_MINS = 1    # integration time in minutes

df_all = blocks_by_month(g, integration_minutes=INTEG_MINS)
df_wet = df_all[df_all['rate'] > 0.1]
years  = sorted(df_wet['year'].unique())

# TODO: build a 2-D matrix of shape (n_years, 12) where each cell holds
#       the YEAR_PCT-th percentile of wet-period rates for that year and month.
#       Use np.nan where there are no wet periods.
# Hint:
# matrix = np.full((len(years), 12), np.nan)
# for i, yr in enumerate(years):
#     df_yr = df_wet[df_wet['year'] == yr]
#     for m in range(1, 13):
#         vals = df_yr.loc[df_yr['month'] == m, 'rate'].values
#         if len(vals) > 0:
#             matrix[i, m-1] = np.percentile(vals, YEAR_PCT)

# TODO: plot as a heatmap using ax.imshow(matrix, ..., cmap='YlOrRd')
#       - x-axis: months (MONTH_NAMES)
#       - y-axis: years
#       - add a colorbar labelled with units
fig, ax = plt.subplots(figsize=(12, max(4, len(years) * 0.45)))
# im = ax.imshow(matrix, aspect='auto', cmap='YlOrRd', interpolation='nearest')
# plt.colorbar(im, ax=ax, label=f'{YEAR_PCT}th-pct rainfall rate (mm hr⁻¹)')
ax.set_xticks(np.arange(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_yticks(np.arange(len(years)))
ax.set_yticklabels(years)
ax.set_xlabel('Month')
ax.set_ylabel('Year')
ax.set_title(
    f'{YEAR_PCT}th-percentile {INTEG_MINS}-min rainfall rate by year and month\n'
    f'{g.label}'
)
plt.tight_layout()
plt.show()
